In [1]:
# ============================================
# 📦 Cell 1: Environment Setup & Imports
# ============================================

import sys
import subprocess

def install_if_missing(package):
    """Installs a package in the current Jupyter environment if not present."""
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Core dependencies
for pkg in ["torch", "torchvision", "diffusers", "transformers", 
            "accelerate", "safetensors", "torchmetrics", "lpips", "Pillow", "numpy"]:
    install_if_missing(pkg)

# ---- Imports ----
import torch, os, time, random
from pathlib import Path
from diffusers import StableDiffusionXLPipeline
from PIL import Image, ImageFilter
import numpy as np
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.psnr import PeakSignalNoiseRatio
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

# ---- Device Configuration ----
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"✅ Using device: {device}")

# ---- Paths ----
BASE_DIR = Path("./evaluation_outputs")
BASE_DIR.mkdir(exist_ok=True)

print("✅ Environment setup complete.")


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installing Pillow ...



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


✅ Using device: mps
✅ Environment setup complete.


In [4]:
# ============================================
# 🎨 Cell 2: Load Quantized Components Manually & Generate Images
# ============================================

from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, AutoencoderKL, EulerDiscreteScheduler
from transformers import CLIPTextModel, CLIPTokenizer
import torch
from pathlib import Path

# ---- Model directory ----
QUANTIZED_MODEL_PATH = "sdxl/quantized_models/sdxl_lightning_2step_mps_fp16"

# ---- Device ----
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

print("⚙️ Loading quantized components individually...")

# ---- Core components ----
scheduler = EulerDiscreteScheduler.from_pretrained(f"{QUANTIZED_MODEL_PATH}/scheduler")
unet = UNet2DConditionModel.from_config(f"{QUANTIZED_MODEL_PATH}/unet")
vae = AutoencoderKL.from_config(f"{QUANTIZED_MODEL_PATH}/vae")

# ---- Substitute missing encoders with pretrained CLIP ----
tokenizer_1 = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch16")
tokenizer_2 = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch16")
text_encoder_1 = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch16")
text_encoder_2 = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch16")

# ---- Build pipeline ----
pipe_quant = StableDiffusionXLPipeline(
    vae=vae,
    text_encoder=text_encoder_1,
    text_encoder_2=text_encoder_2,
    tokenizer=tokenizer_1,
    tokenizer_2=tokenizer_2,
    unet=unet,
    scheduler=scheduler
).to(device)

pipe_quant.enable_attention_slicing()
print("✅ Quantized SDXL pipeline initialized successfully (config-only fallback).")

# ---- Prompt and generation ----
prompt = "A futuristic city skyline at night with glowing neon lights"
generator = torch.manual_seed(1234)

print("🧠 Generating image from quantized model config...")
image = pipe_quant(
    prompt=prompt,
    num_inference_steps=4,
    guidance_scale=0,
    generator=generator,
    height=512,
    width=512
).images[0]

Path("evaluation_outputs").mkdir(exist_ok=True)
image.save("evaluation_outputs/quantized_output.png")
print("✅ Image generated and saved to evaluation_outputs/quantized_output.png")

⚙️ Loading quantized components individually...
✅ Quantized SDXL pipeline initialized successfully (config-only fallback).
🧠 Generating image from quantized model config...


AttributeError: 'NoneType' object has no attribute 'repeat'